<a href="https://colab.research.google.com/github/ethanlam101/151B_SP26_Competition_deep_learners/blob/main/starter_code_cse151b_comp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [1]:
# lowk dont use

# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

### Run the cell below every time to activate the installed environment.

In [2]:
# # activate venv after installation. This needs to be run everytime.
# !source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [1]:
import subprocess, sys

# Install exact versions known to work together on Colab T4
!pip install \
    vllm==0.8.5 \
    transformers==4.51.3 \
    protobuf==5.29.6 \
    triton==3.0.0 \
    antlr4-python3-runtime==4.11.1

print("Done! Restart runtime before continuing.")

  Using cached triton-3.0.0-1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.3 kB)
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
  Using cached outlines-0.1.11-py3-none-any.whl.metadata (17 kB)
  Using cached compressed_tensors-0.9.3-py3-none-any.whl.metadata (7.0 kB)
  Using cached vllm-0.8.5-cp38-abi3-manylinux1_x86_64.whl.metadata (14 kB)
ERROR: Cannot install torch==2.6.0 and triton==3.0.0 because these package versions have conflicting dependencies.

The conflict is caused by:
    The user requested triton==3.0.0
    torch 2.6.0 depends on triton==3.2.0; platform_system == "Linux" and platform_machine == "x86_64"

To fix this you could try to:
1. loosen the range of package versions you've specified
2. remove package versions to allow pip to attempt to solve the dependency conflict

ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/

In [1]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0" #"1"                    # CUDA_VISIBLE_DEVICES
# DATA_PATH   = "data/public.jsonl"
DATA_PATH   = "public.jsonl"

OUTPUT_PATH = "results/starter_results.jsonl"
# MAX_TOKENS  = 32768
MAX_TOKENS = 4096

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

INFO 05-01 07:42:50 [__init__.py:239] Automatically detected platform cuda.


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [2]:
import pandas as pd

In [3]:
!wget https://raw.githubusercontent.com/ethanlam101/151B_SP26_Competition_deep_learners/refs/heads/main/data/public.jsonl

--2026-05-01 06:32:53--  https://raw.githubusercontent.com/ethanlam101/151B_SP26_Competition_deep_learners/refs/heads/main/data/public.jsonl
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 668383 (653K) [text/plain]
Saving to: ‘public.jsonl’

public.jsonl        100%[===================>] 652.72K  --.-KB/s    in 0.006s  

2026-05-01 06:32:53 (109 MB/s) - ‘public.jsonl’ saved [668383/668383]



In [3]:
df = pd.read_json("public.jsonl", lines=True)
df

,question,answer,id,options
0,Find the sum of the first $325$ positive even ...,[325*(1+325)],0,NaN
1,$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ...,F,1,"[$0$, $frac{1}{a}$, $frac{3}{a}$, $frac{1}{2a^..."
2,A roasted turkey is taken from an oven when it...,"[143.224229233795, 2.32624773420025]",2,NaN
3,Reduce the fraction ${\frac{25}{40}}$. [ANS],[5/8],3,NaN
4,"Given $u(x, y) = x^3 + 6x^2y - 3xy^2 - 2y^3$, ...",C,4,"[$$\n( 6+4 \mathrm{i} ) z^{5}\n$$, $$\n( 4-5 \..."
...,...,...,...,...
1121,Give the constant term and the coefficient of ...,"[0, 10+r]",1121,NaN
1122,Find all angles $x$ (in radians) that satisfy ...,[2*pi/3+k*pi],1122,NaN
1123,Divide the integer by the fraction:\n${15 \div...,[24],1123,NaN
1124,(a) What is the average rate of change of $g(x...,"[5, increasing]",1124,NaN


In [4]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [26]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [8]:
# SO THE NEXT CELL WORKS: DO RUNTIME --> CHANGE RUNTIME TYPE --> CHANGE TO 64 GPU

In [6]:
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    # quantization="bitsandbytes",
    # load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.9, #originally 0.50
    max_model_len=5120, #originally 16384
    trust_remote_code=True,
    max_num_seqs=16, #originally 256
    max_num_batched_tokens=32768,
    dtype="half",
)

tokenizer = llm.get_tokenizer()
# tokenizer.pad_token = tokenizer.eos_token

sampling_params = SamplingParams(
    max_tokens=2048, #originally MAX_TOKENS
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


WARNING 05-01 07:43:13 [config.py:2972] Casting torch.bfloat16 to torch.float16.
INFO 05-01 07:43:34 [config.py:717] This model supports multiple tasks: {'classify', 'embed', 'score', 'generate', 'reward'}. Defaulting to 'generate'.
WARNING 05-01 07:43:34 [arg_utils.py:1658] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
INFO 05-01 07:43:34 [llm_engine.py:240] Initializing a V0 LLM engine (v0.8.5) with config: model='Qwen/Qwen3-4B-Thinking-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Thinking-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=5120, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 05-01 07:44:14 [loader.py:458] Loading weights took 33.77 seconds
INFO 05-01 07:44:15 [model_runner.py:1140] Model loading took 7.6065 GiB and 35.427572 seconds
INFO 05-01 07:44:31 [worker.py:287] Memory profiling takes 15.52 seconds
INFO 05-01 07:44:31 [worker.py:287] the current vLLM instance can use total_gpu_memory (14.56GiB) x gpu_memory_utilization (0.90) = 13.11GiB
INFO 05-01 07:44:31 [worker.py:287] model weights take 7.61GiB; non_torch_memory takes 0.05GiB; PyTorch activation peak memory takes 2.41GiB; the rest of the memory reserved for KV Cache is 3.04GiB.
INFO 05-01 07:44:31 [executor_base.py:112] # cuda blocks: 1382, # CPU blocks: 1820
INFO 05-01 07:44:31 [executor_base.py:117] Maximum concurrency for 5120 tokens per request: 4.32x
INFO 05-01 07:44:34 [model_runner.py:1450] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.

Capturing CUDA graph shapes:   0%|          | 0/5 [00:00<?, ?it/s]

INFO 05-01 07:44:43 [model_runner.py:1592] Graph capturing finished in 9 secs, took 0.08 GiB
INFO 05-01 07:44:43 [llm_engine.py:437] init engine (profile, create kv cache, warmup model) took 28.04 seconds
Model loaded.


## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [8]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# llm = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     trust_remote_code=True,
#     quantization_config=bnb_config,
#     device_map="auto",
# )


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [13]:
# Build prompts for first 5 entries
prompts = []
for item in data[:5]:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user + "/no_think"}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 5 questions...


Processed prompts:   0%|          | 0/5 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


── Response 0 (id=0) ──
Okay, let's see. The problem is to find the sum of the first 325 positive even whole numbers. Hmm, first, I need to remember what the first positive even whole numbers are. Let me think. The positive even whole numbers start from 2, right? So the first one is 2, the second is 4, the third is 6, and so on. So it's 2, 4, 6, 8, ..., up to the 325th term.

I need to find the sum of this sequence. Sinc ...

── Response 1 (id=1) ──
Okay, let's try to solve this integral. The problem is the integral from negative infinity to positive infinity of (a^(3/2)) divided by (s² + a²) ds. Hmm, first, I need to remember how to integrate functions like 1/(s² + a²). 

Wait, the integral of 1/(s² + a²) ds is (1/a) arctan(s/a) + C, right? Because the derivative of arctan(s/a) is (1/(1 + (s/a)²)) * (1/a) = a/(a² + s²), so multiplying by 1/a ...

── Response 2 (id=2) ──
Okay, let's tackle part (a) first. I remember that cooling problems like this usually follow Newton's Law of Coolin

### Generate with Transformers (for Datahub)

In [8]:
# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # Tokenize (padded batch)
# print(f"Generating responses for {len(prompts)} questions...")
# inputs = tokenizer(
#     prompts,
#     return_tensors="pt",
#     padding=True,
#     truncation=True,
#     max_length=16384,
# ).to(llm.device)

# # Generate
# with torch.no_grad():
#     output_ids = llm.generate(
#         **inputs,
#         max_new_tokens=MAX_TOKENS,
#         temperature=0.6,
#         top_p=0.95,
#         top_k=20,
#         repetition_penalty=1.0,
#         do_sample=True,
#     )

# # Decode only the new tokens (strip the prompt)
# responses = []
# for i, out in enumerate(output_ids):
#     new_tokens = out[inputs["input_ids"].shape[1]:]
#     responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [11]:
!wget https://raw.githubusercontent.com/ethanlam101/151B_SP26_Competition_deep_learners/refs/heads/main/judger.py

--2026-05-01 07:27:45--  https://raw.githubusercontent.com/ethanlam101/151B_SP26_Competition_deep_learners/refs/heads/main/judger.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 38572 (38K) [text/plain]
Saving to: ‘judger.py.2’

judger.py.2         100%[===================>]  37.67K  --.-KB/s    in 0s      

2026-05-01 07:27:46 (142 MB/s) - ‘judger.py.2’ saved [38572/38572]



In [11]:
!wget https://raw.githubusercontent.com/ethanlam101/151B_SP26_Competition_deep_learners/refs/heads/main/utils.py

--2026-05-01 07:14:46--  https://raw.githubusercontent.com/ethanlam101/151B_SP26_Competition_deep_learners/refs/heads/main/utils.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 11546 (11K) [text/plain]
Saving to: ‘utils.py.1’

utils.py.1          100%[===================>]  11.28K  --.-KB/s    in 0s      

2026-05-01 07:14:46 (89.1 MB/s) - ‘utils.py.1’ saved [11546/11546]



In [23]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring:   0%|          | 5/1126 [00:00<03:07,  5.98it/s]

Scoring complete. 5 results.


In [24]:
results

[{'id': 0,
  'is_mcq': False,
  'gold': ['325*(1+325)'],
  'response': 'Okay, let\'s see. The problem is to find the sum of the first 325 positive even whole numbers. Hmm, first, I need to remember what the first positive even whole numbers are. Let me think. The positive even whole numbers start from 2, right? So the first one is 2, the second is 4, the third is 6, and so on. So it\'s 2, 4, 6, 8, ..., up to the 325th term.\n\nI need to find the sum of this sequence. Since it\'s an arithmetic sequence, maybe I can use the formula for the sum of an arithmetic series. Let me recall. The sum of the first n terms of an arithmetic sequence is given by S_n = n/2 * (a_1 + a_n), where a_1 is the first term and a_n is the nth term. Alternatively, it\'s also S_n = n * a_1 + n(n - 1)/2 * d, where d is the common difference. Let\'s see which one is easier here.\n\nFirst, let\'s confirm the sequence. The first term a_1 is 2. The common difference d is 2, because each term is 2 more than the previou

## 8. Summary

Print accuracy broken down by question type.

In [25]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :    0 /    2  (0.00%)
  Free-form  :    1 /    3  (33.33%)
  Overall    :    1 /    5  (20.00%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!